In [1]:
import sagemaker
import boto3
from sagemaker.amazon.amazon_estimator import get_image_uri
from sagemaker.session import s3_input, Session

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[01/10/25 21:45:02] INFO     Found credentials from IAM Role:                                   ]8;id=428401;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=817941;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
bucket_name = "sagemakerprojectbucket2".lower()  # Bucket name in lowercase
my_region = boto3.session.Session().region_name
print("Region:", my_region)

Region: us-east-1


In [14]:
import sagemaker
sagemaker_role = sagemaker.get_execution_role()

In [15]:
print("Role :",sagemaker_role )

Role : arn:aws:iam::396608801529:role/sagemakeraccess


In [3]:
s3 = boto3.resource('s3')
try:
    if my_region == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': my_region}
        )
    print("S3 bucket created successfully")
except Exception as e:
    print("S3 error:", e)

                    INFO     Found credentials from IAM Role:                                   ]8;id=742114;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=42365;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

S3 bucket created successfully


In [4]:
prefix = 'xgboost-as-a-built-in-algo'

In [5]:
output_path = f"s3://{bucket_name}/{prefix}/output"
print(output_path)

s3://sagemakerprojectbucket2/xgboost-as-a-built-in-algo/output


In [6]:
import pandas as pd
import urllib
try:
    urllib.request.urlretrieve ("https://d1.awsstatic.com/tmt/build-train-deploy-machine-learning-model-sagemaker/bank_clean.27f01fbbdf43271788427f3682996ae29ceca05d.csv", "bank_clean.csv")
    print('Success: downloaded bank_clean.csv.')
except Exception as e:
    print("Data Loading Error :", e)
    
try:
    model_data = pd.read_csv('./bank_clean.csv', index_col=0)
    print("Success loading the dataset")
except Exception as e:
    print("Data load error :", e)

Success: downloaded bank_clean.csv.
Success loading the dataset


In [7]:
# Train Test Split

import numpy as np
train_data, test_data = np.split(model_data.sample(frac=1, random_state =101), [int(0.7 * len(model_data))])
print(f"Train Data Shape = {train_data.shape}")
print(f"Test Data Shape = {test_data.shape}")

Train Data Shape = (28831, 61)
Test Data Shape = (12357, 61)


In [9]:
# Remeber in sagemaker you always put Target column first
# Test Data in s3
import os
from sagemaker.inputs import TrainingInput 

pd.concat([train_data['y_yes'], train_data.drop(['y_yes','y_no'], axis=1)], axis=1).to_csv('train.csv', index=False, header=False)

boto3.Session().resource('s3').Bucket(bucket_name).Object(os.path.join(prefix,'train/train.csv')).upload_file('train.csv')
s3_input_train = TrainingInput(
    s3_data='s3://{}/{}/train'.format(bucket_name, prefix),  # Specify the S3 path
    content_type='csv'  # Adjust the content type if needed (e.g., 'text/csv' or other formats)
)

[01/10/25 23:03:42] INFO     Found credentials from IAM Role:                                   ]8;id=772625;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=163623;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [10]:
# Test Data Into s3
from sagemaker.inputs import TrainingInput  

pd.concat([test_data['y_yes'], test_data.drop(['y_no', 'y_yes'], axis=1)], axis=1).to_csv('test.csv', index=False, header=False)
boto3.Session().resource('s3').Bucket(bucket_name).Object(os.path.join(prefix, 'test/test.csv')).upload_file('test.csv')

s3_input_test = TrainingInput(
    s3_data='s3://{}/{}/test'.format(bucket_name, prefix),  # Specify the S3 path for test data
    content_type='csv'  # Adjust the content type if needed
)

[01/10/25 23:04:47] INFO     Found credentials from IAM Role:                                   ]8;id=974633;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=37514;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/botocore/credentials.py#1075\1075]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

In [18]:
container = get_image_uri(boto3.Session().region_name,
                         'xgboost',
                         repo_version='1.5-1'
                         )

[01/10/25 23:31:03] WARNING  The method get_image_uri has been renamed in sagemaker>=2.          ]8;id=299105;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/deprecations.py\deprecations.py]8;;\:]8;id=91314;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/deprecations.py#34\34]8;;\
                             See: https://sagemaker.readthedocs.io/en/stable/v2.html for                           
                             details.                                                                              

                    INFO     Ignoring unnecessary instance type: None.                            ]8;id=712470;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=402295;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/image_uris.py#528\528]8;;\

In [19]:
hyperparameters = {
    "max_depth": "5",                # Maximum depth of a tree. Try 3, 5, 7, 10
    "eta": "0.2",                    # Learning rate. Try values like 0.01, 0.1, 0.2, 0.3
    "gamma": "4",                    # Minimum loss reduction required to make a further partition on a leaf node.
    "min_child_weight": "6",         # Minimum sum of instance weight (hessian) needed in a child.
    "subsample": "0.7",              # Fraction of training samples used for growing trees. Try 0.5, 0.7, 1.0
    "colsample_bytree": "0.8",       # Fraction of features to be used for building each tree.
    "objective": "binary:logistic",  # Type of model (binary classification in this case).
    "num_round": "100",              # Number of boosting rounds (iterations). Try 50, 100, 200
    "lambda": "1",                   # L2 regularization term (use to prevent overfitting).
    "alpha": "0.5",                  # L1 regularization term (use to prevent overfitting).
    "eval_metric": "auc",            # Metric used for evaluation (AUC for classification tasks).
    "scale_pos_weight": "1",         # Balances the positive and negative classes (useful for imbalanced classes).
    "tree_method": "hist",           # Algorithm used for building trees (hist is fast for large datasets).
    "early_stopping_rounds": "10"    # Number of rounds without improvement to stop training early (useful for avoiding overfitting).
}


In [20]:
from sagemaker.estimator import Estimator

# Correct the arguments and use 'image_uri' instead of 'image_name'
estimator = Estimator(
    image_uri=container,                # Use 'image_uri' for the container image
    hyperparameters=hyperparameters,   # Pass the hyperparameters dictionary
    role=sagemaker_role,               # IAM role for SageMaker execution
    instance_count=1,                  # Number of training instances
    instance_type='ml.m5.2xlarge',     # Instance type
    volume_size=5,                     # Volume size in GB
    output_path=output_path,           # S3 path to save model artifacts
    use_spot_instances=True,           # Use spot instances to save costs
    max_run=300,                       # Maximum training time in seconds
    max_wait=600                       # Maximum waiting time for spot instances
)


In [21]:
estimator.fit({'train':s3_input_train,'validation':s3_input_test})

[01/10/25 23:31:15] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=832657;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=426630;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#90\90]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=738908;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=264932;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             sagemaker-xgboost-2025-01-10-23-31-15-115                                             

2025-01-10 23:31:17 Starting - Starting the training job...
2025-01-10 23:31:32 Starting - Preparing the instances for training...
2025-01-10 23:32:18 Downloading - Downloading the training image......
2025-01-10 23:32:58 Training - Training image download completed. Training in progress./miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2025-01-10 23:33:15.650 ip-10-0-159-120.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2025-01-10 23:33:15.671 ip-10-0-159-120.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2025-01-10:23:33:16:INFO] Imported framework sagemaker_xgboost_container.training
[2025-01-10:23:33:16:INFO] Failed to parse hyperparameter eval_metric value auc to Json.
Returning the value itself
[2025-01-10:23:33:16:IN

### Deployment

In [22]:
xgb_predictor = estimator.deploy(initial_instance_count=1,instance_type='ml.m4.xlarge')

[01/10/25 23:39:39] INFO     Creating model with name: sagemaker-xgboost-2025-01-10-23-39-39-625    ]8;id=368253;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=953452;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4094\4094]8;;\

[01/10/25 23:39:40] INFO     Creating endpoint-config with name                                     ]8;id=810974;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=181260;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#5889\5889]8;;\
                             sagemaker-xgboost-2025-01-10-23-39-39-625                                             

                    INFO     Creating endpoint with name sagemaker-xgboost-2025-01-10-23-39-39-625  ]8;id=323784;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=60001;file:///home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/session.py#4711\4711]8;;\

--------!

### Prediction

In [28]:
from sagemaker.serializers import CSVSerializer  
import numpy as np

test_data_array = test_data.drop(['y_no', 'y_yes'], axis=1).values #load the data into an array

xgb_predictor.content_type = 'text/csv' # set the data type for an inference
xgb_predictor.serializer = CSVSerializer() # set the serializer type

predictions = xgb_predictor.predict(test_data_array).decode('utf-8') # predict!

# Remove any unwanted characters (such as leading/trailing spaces or newlines)
predictions_cleaned = predictions.strip()

# Split the cleaned predictions string into a list using newline as the separator
predictions_list = predictions_cleaned.split('\n')

# Convert the list of predictions into a NumPy array
predictions_array = np.array([float(pred) for pred in predictions_list])

# Check the shape of the resulting array
print("Predictions array shape:", predictions_array.shape)
print("Predictions array:\n", predictions_array)

Predictions array shape: (12357,)
Predictions array:
 [0.09581964 0.08374396 0.08825492 ... 0.05032252 0.04752673 0.08591904]


In [29]:
predictions_array

array([0.09581964, 0.08374396, 0.08825492, ..., 0.05032252, 0.04752673,
       0.08591904])

In [31]:
import pandas as pd
import numpy as np

# Assuming predictions_array and test_data['y_yes'] are already available

# Create confusion matrix
cm = pd.crosstab(index=test_data['y_yes'], columns=np.round(predictions_array), rownames=['Observed'], colnames=['Predicted'])

# Extract values from the confusion matrix
tn = cm.iloc[0, 0]  # True Negatives
fn = cm.iloc[1, 0]  # False Negatives
tp = cm.iloc[1, 1]  # True Positives
fp = cm.iloc[0, 1]  # False Positives

# Calculate Overall Classification Rate
p = (tp + tn) / (tp + tn + fp + fn) * 100

# Print Overall Classification Rate
print("\n{0:<20}{1:<4.1f}%".format("Overall Classification Rate: ", p))
print('\n')

# Print the confusion matrix metrics
print("{0:<15}{1:<15}{2:>8}".format("Predicted", "No Purchase", "Purchase"))
print("Observed")
print("{0:<15}{1:<2.0f}% ({2:<}){3:>6.0f}% ({4:<})".format("No Purchase", tn / (tn + fn) * 100, tn, fp / (tp + fp) * 100, fp))
print("{0:<16}{1:<1.0f}% ({2:<}){3:>7.0f}% ({4:<}) \n".format("Purchase", fn / (tn + fn) * 100, fn, tp / (tp + fp) * 100, tp))


Overall Classification Rate: 89.6%


Predicted      No Purchase    Purchase
Observed
No Purchase    90% (10797)    32% (127)
Purchase        10% (1159)     68% (274) 

